In [ ]:
# 1. Install Unsloth and core dependencies
!pip install unsloth
!pip install --no-deps "xformers<0.0.28" "trl<0.10.0" peft accelerate bitsandbytes

# 2. Install Vision & Data dependencies
!pip install qwen-vl-utils datasets wandb

# 3. CRITICAL: Force install the latest transformers from GitHub to prevent Qwen2_VL mapping errors
!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
!pip uninstall -y trl
!pip install git+https://github.com/huggingface/trl.git

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [2]:
# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

import re
def extract_json(text):
    """Safely extracts JSON from model completions that might include markdown."""
    text = text.strip()
    json_match = re.search(r'```json\n(.*?)\n```', text, re.DOTALL)
    if json_match:
        text = json_match.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None
    

def robust_float(val):
    """
    Safely casts strings, integers, and scientific notation (3e+12) to floats.
    Strips LLM hallucinations like stray spaces or commas.
    """
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        try:
            # Clean common LLM formatting artifacts
            clean_val = val.strip().replace(',', '')
            # Python handles "3e+12" and "-2.45e-10" natively
            return float(clean_val)
        except ValueError:
            raise ValueError(f"Cannot parse {val} to float")
    raise ValueError("Not a valid type for float conversion")

In [3]:
# ==========================================
# 2. REWARD FUNCTIONS
# ==========================================

def format_reward_func(completions, **kwargs):
    """Reward 1: Does the output compile as valid JSON? (The Gatekeeper)"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        # Heavy penalty for breaking syntax; this is a baseline requirement.
        rewards.append(1.0 if parsed is not None else -2.0)
    return rewards


def schema_enforcement_reward_func(completions, **kwargs):
    """Reward 2: Are the top-level keys correct and forbidden keys absent?"""
    rewards = []
    required_keys = {"title", "panel_count",
                     "panel_layout", "panels", "chart_type"}
    forbidden_keys = {"math", "rel", "stats", "trend"}

    for comp in completions:
        parsed = extract_json(comp)
        if not parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        # +0.5 for hitting all required root keys
        if required_keys.issubset(parsed.keys()):
            score += 0.5

        # -1.0 penalty if the model hallucinates legacy token-heavy semantic fields
        comp_str = str(comp).lower()
        if any(f'"{fk}"' in comp_str for fk in forbidden_keys):
            score -= 1.0

        rewards.append(score)
    return rewards


def panel_architecture_reward_func(completions, **kwargs):
    """Reward 3: Does the panel math make logical sense?"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        if not parsed or "panels" not in parsed or "panel_count" not in parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        try:
            # Does the stated count match the actual array length?
            actual_count = len(parsed["panels"])
            if actual_count == parsed["panel_count"]:
                score += 0.5

            # Does the layout grid support the panel count? (e.g., [2,1] grid supports up to 2 panels)
            if "panel_layout" in parsed and len(parsed["panel_layout"]) == 2:
                grid_capacity = parsed["panel_layout"][0] * \
                    parsed["panel_layout"][1]
                if grid_capacity >= actual_count:
                    score += 0.5
        except (TypeError, ValueError):
            pass

        rewards.append(score)
    return rewards


def axis_mapping_reward_func(completions, **kwargs):
    """Reward 4: Are axes structurally sound before checking series data?"""
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        if not parsed or "panels" not in parsed:
            rewards.append(0.0)
            continue

        score = 0.0
        total_panels = len(parsed["panels"])
        valid_panels = 0

        for panel in parsed["panels"]:
            if "axes" in panel and isinstance(panel["axes"], list):
                # Check if at least an x_axis and y_axis are defined properly
                axes_names = [ax.get("name")
                              for ax in panel["axes"] if isinstance(ax, dict)]
                if "x_axis" in axes_names and "y_axis" in axes_names:
                    valid_panels += 1

        # Normalize score based on how many panels have valid axis definitions
        if total_panels > 0:
            score += (valid_panels / total_panels) * 1.0

        rewards.append(score)
    return rewards


def evaluate_bar_series(parsed_panel, target_panel):
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # A. Check Categorical X
            if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                data_score += 0.2

            # B. Check Primary Y Value
            try:
                p_val, t_val = robust_float(p_pt[1]), robust_float(t_pt[1])
                tol = max(0.05 * abs(t_val), 0.01)
                if abs(p_val - t_val) <= tol:
                    data_score += 0.5
                elif abs(p_val - t_val) <= tol * 2:
                    data_score += 0.2
            except ValueError:
                pass

            # C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2)
            for i in range(2, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_val_err, t_val_err = robust_float(
                        p_pt[i]), robust_float(t_pt[i])
                    tol_err = max(0.05 * abs(t_val_err), 0.01)
                    if abs(p_val_err - t_val_err) <= tol_err:
                        data_score += 0.25
                except ValueError:
                    # Fallback: It's a text marker (e.g., "Point A")
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += 0.25

        score += data_score / max(1, len(t_data))
    return score


def evaluate_line_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Line and Area charts (Standard, Stacked, Errorband).
    Dynamically handles both 2D [x, y] coordinates and 3D [x, y_max, y_min] boundaries.
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward: Series count match
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Points
    for p_s, t_s in zip(parsed_ser, target_ser):

        # Reward correct series mapping
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # --- A. Check X-Axis Value (Index 0: Hybrid Categorical/Numerical) ---
            try:
                # Attempt numerical matching first
                p_x = robust_float(p_pt[0])
                t_x = robust_float(t_pt[0])
                tol_x = max(0.05 * abs(t_x), 0.01)

                if abs(p_x - t_x) <= tol_x:
                    data_score += 0.2
            except (ValueError, TypeError):
                # Fallback to categorical string matching (e.g., "January", "QuickSort")
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    data_score += 0.2

            # --- B. Check Primary Y-Axis Value (Index 1) ---
            try:
                p_y = robust_float(p_pt[1])
                t_y = robust_float(t_pt[1])
                tol_y = max(0.05 * abs(t_y), 0.01)

                if abs(p_y - t_y) <= tol_y:
                    data_score += 0.5
                elif abs(p_y - t_y) <= tol_y * 2:
                    data_score += 0.2  # Partial credit for being close
            except (ValueError, TypeError):
                pass

            # --- C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2) ---
            for i in range(2, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_val_extra, t_val_extra = robust_float(
                        p_pt[i]), robust_float(t_pt[i])
                    tol_extra = max(0.05 * abs(t_val_extra), 0.01)
                    if abs(p_val_extra - t_val_extra) <= tol_extra:
                        data_score += 0.3
                    elif abs(p_val_extra - t_val_extra) <= tol_extra * 2:
                        data_score += 0.1
                except ValueError:
                    # Fallback: It's a text marker (e.g., "Point A")
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += 0.3

        # Normalize to prevent massive arrays from disproportionately skewing the RL objective
        normalized_data_score = data_score / max(1, len(t_data))
        score += normalized_data_score

    return score


def evaluate_box_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Box plots.
    Dynamically handles both 5-number summary arrays (length 6) and individual outlier points (length 2).
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward: Series count match (Usually 1 for summary, 1 for outliers)
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Points
    for p_s, t_s in zip(parsed_ser, target_ser):

        # Reward correct series mapping ("Box Distribution" vs "Outliers")
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # Reward finding the outlier flag correctly
        if t_s.get("is_outlier") == 1 and p_s.get("is_outlier") == 1:
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        data_score = 0.0
        # Check for empty series
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if not p_data or not t_data:
            continue

        # --- THE ANTI-LOOPING & TRUNCATION PENALTY ---
        # Punish the model for missing points or hallucinating extra looped points
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            # -0.1 penalty for every hallucinated or missing coordinate
            data_score -= (0.1 * length_diff)
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            # --- A. Check Categorical X-Axis Value (Index 0) ---
            try:
                # Attempt numerical matching first (rare for boxplots, but safe)
                p_x = robust_float(p_pt[0])
                t_x = robust_float(t_pt[0])
                tol_x = max(0.05 * abs(t_x), 0.01)

                if abs(p_x - t_x) <= tol_x:
                    data_score += 0.2
            except (ValueError, TypeError):
                # Fallback to categorical string matching (e.g., "E.coli")
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    data_score += 0.2

            # --- B. Dynamically Check Y-Axis Values (Indices 1 through End) ---
            # If t_pt has length 6 (5-num summary), each stat is worth 0.2 (total 1.0)
            # If t_pt has length 2 (outlier), the single stat is worth 1.0
            # --- B. Dynamically Check Y-Axis Values (Indices 1 through End) ---
            stat_weight = 1.0 / (len(t_pt) - 1)

            for i in range(1, len(t_pt)):
                if i >= len(p_pt):
                    break

                try:
                    p_val, t_val = robust_float(p_pt[i]), robust_float(t_pt[i])
                    tol_val = max(0.05 * abs(t_val), 0.01)

                    if abs(p_val - t_val) <= tol_val:
                        data_score += stat_weight
                    elif abs(p_val - t_val) <= tol_val * 2:
                        data_score += stat_weight * 0.4
                except ValueError:
                    # Fallback for text markers in outlier arrays (e.g. [Cat, Value, "Outlier A"])
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += stat_weight

        # Normalize the data score by the target array length.
        normalized_data_score = data_score / max(1, len(t_data))
        score += normalized_data_score

    return score


def evaluate_histogram_series(parsed_panel, target_panel):
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # 1. Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    # 2. Evaluate Data Structures
    for p_s, t_s in zip(parsed_ser, target_ser):
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # --- PATH A: KDE Density Curves ---
        if t_s.get("is_kde") == 1 or t_s.get("type") == "density_curve":
            if p_s.get("is_kde") == 1:
                score += 0.1

            t_peak, p_peak = t_s.get(
                "exact_peak", {}), p_s.get("exact_peak", {})
            if isinstance(t_peak, dict) and isinstance(p_peak, dict) and t_peak:
                try:
                    for k in ["x", "y"]:
                        p_val, t_val = robust_float(p_peak.get(
                            k, 0)), robust_float(t_peak.get(k, 0))
                        if abs(p_val - t_val) <= max(0.05 * abs(t_val), 0.01):
                            score += 0.2
                except ValueError:
                    pass

            # Route to "points" array
            p_data, t_data = p_s.get("points", []), t_s.get("points", [])

        # --- PATH B: Aggregated & Normal Bins ---
        else:
            if t_s.get("is_aggregated") == 1 and p_s.get("is_aggregated") == 1:
                score += 0.1

            # Route to "data" array
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])

        # 3. Evaluate the Numerics
        if not p_data or not t_data:
            continue

        # ---------------------------------------------------------
        # THE ANTI-LOOPING & TRUNCATION PENALTY
        # Punish the model for missing points or hallucinating extra points.
        # This works perfectly for both 25-bin arrays and 150-point KDE curves.
        # ---------------------------------------------------------
        length_diff = abs(len(p_data) - len(t_data))
        data_score = 0.0

        if length_diff > 0:
            data_score -= (0.1 * length_diff)

        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue

            pt_weight = 1.0 / len(t_pt)

            for i in range(len(t_pt)):
                if i >= len(p_pt):
                    break

                try:
                    p_val, t_val = robust_float(p_pt[i]), robust_float(t_pt[i])
                    tol_val = max(0.05 * abs(t_val), 0.01)

                    if abs(p_val - t_val) <= tol_val:
                        data_score += pt_weight
                    elif abs(p_val - t_val) <= tol_val * 2:
                        data_score += pt_weight * 0.4
                except ValueError:
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        data_score += pt_weight

        score += data_score / max(1, len(t_data))

    return score


def evaluate_scatter_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Scatter plots.
    Dynamically handles standard [x, y] coordinates and heavily compressed 
    k-means cluster descriptions (centroids, bounding boxes, outliers).
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        # 1. Base Metadata
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        # Reward finding the correct secondary axis mapping (Crucial for multi-axes charts)
        if str(p_s.get("y_ref")) == str(t_s.get("y_ref")):
            score += 0.2

        # ---------------------------------------------------------
        # PATH A: Clustered Scatter (Spatial & Density Math)
        # ---------------------------------------------------------
        if t_s.get("is_clustered") == 1:
            if p_s.get("is_clustered") == 1:
                score += 0.2

            t_clusters = t_s.get("cluster_descriptions", [])
            p_clusters = p_s.get("cluster_descriptions", [])

            cluster_score = 0.0
            for p_c, t_c in zip(p_clusters, t_clusters):
                if not isinstance(p_c, dict) or not isinstance(t_c, dict):
                    continue

                # A1. Score Centroids
                try:
                    p_cx, t_cx = robust_float(p_c.get("centroid", [0, 0])[0]), robust_float(
                        t_c.get("centroid", [0, 0])[0])
                    p_cy, t_cy = robust_float(p_c.get("centroid", [0, 0])[1]), robust_float(
                        t_c.get("centroid", [0, 0])[1])

                    if abs(p_cx - t_cx) <= max(0.05 * abs(t_cx), 0.01):
                        cluster_score += 0.2
                    if abs(p_cy - t_cy) <= max(0.05 * abs(t_cy), 0.01):
                        cluster_score += 0.2
                except (ValueError, TypeError, IndexError):
                    pass

                # A2. Score Bounding Boxes
                p_bb, t_bb = p_c.get("bounding_box", {}), t_c.get(
                    "bounding_box", {})
                for k in ["x_min", "x_max", "y_min", "y_max"]:
                    try:
                        if abs(robust_float(p_bb.get(k)) - robust_float(t_bb.get(k))) <= max(0.05 * abs(robust_float(t_bb.get(k))), 0.01):
                            cluster_score += 0.1
                    except (ValueError, TypeError):
                        pass

                # A3. Score Density Metadata (Critical for Stage 2 Claims)
                try:
                    if abs(robust_float(p_c.get("density_percentage")) - robust_float(t_c.get("density_percentage"))) <= 1.0:
                        cluster_score += 0.2
                except (ValueError, TypeError):
                    pass

            score += cluster_score / max(1, len(t_clusters))

            # For clustered charts, the discrete points are stored in "outliers"
            p_data = p_s.get("outliers", [])
            t_data = t_s.get("outliers", [])

        # ---------------------------------------------------------
        # PATH B: Standard Scatter (Inside evaluate_scatter_series)
        # ---------------------------------------------------------
        else:
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])
            if not p_data or not t_data:
                continue

            data_score = 0.0
            for p_pt, t_pt in zip(p_data, t_data):
                if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                    continue

                # Check X
                try:
                    if abs(robust_float(p_pt[0]) - robust_float(t_pt[0])) <= max(0.05 * abs(robust_float(t_pt[0])), 0.01):
                        data_score += 0.2
                except ValueError:
                    if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                        data_score += 0.2

                # Check Y
                try:
                    if abs(robust_float(p_pt[1]) - robust_float(t_pt[1])) <= max(0.05 * abs(robust_float(t_pt[1])), 0.01):
                        data_score += 0.5
                    elif abs(robust_float(p_pt[1]) - robust_float(t_pt[1])) <= max(0.10 * abs(robust_float(t_pt[1])), 0.02):
                        data_score += 0.2
                except ValueError:
                    pass

                # --- C. Dynamic Check for Error Bounds OR Text Markers (Indices >= 2) ---
                for i in range(2, len(t_pt)):
                    if i >= len(p_pt):
                        break
                    try:
                        p_val_extra, t_val_extra = robust_float(
                            p_pt[i]), robust_float(t_pt[i])
                        tol_extra = max(0.05 * abs(t_val_extra), 0.01)
                        if abs(p_val_extra - t_val_extra) <= tol_extra:
                            data_score += 0.2
                    except ValueError:
                        # Fallback: It's a text marker
                        if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                            data_score += 0.2

            score += data_score / max(1, len(t_data))

    return score


def evaluate_pie_series(parsed_panel, target_panel):
    """
    Scores data accuracy for Pie and Donut charts.
    Evaluates exact percentage extractions with tight tolerances, 
    and optional absolute value pairings.
    """
    score = 0.0
    parsed_ser = parsed_panel.get("ser", [])
    target_ser = target_panel.get("ser", [])

    if not target_ser:
        return 0.0

    # Architecture Reward
    if len(parsed_ser) == len(target_ser):
        score += 0.5
    else:
        score -= 0.5 * abs(len(parsed_ser) - len(target_ser))

    for p_s, t_s in zip(parsed_ser, target_ser):
        # Match the slice label (often stored in the 'name' field)
        if str(p_s.get("name")).lower() == str(t_s.get("name")).lower():
            score += 0.2

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])

        if not p_data or not t_data:
            continue

        # Pie charts usually have a single inner array per series definition: data[0]
        p_pt = p_data[0] if isinstance(p_data[0], list) else []
        t_pt = t_data[0] if isinstance(t_data[0], list) else []

        if not p_pt or not t_pt:
            continue

        # 1. Verify Percentage (Index 0)
        # Using a tighter 2% tolerance here since slices are highly dependent on exact ratios
        try:
            p_pct = robust_float(p_pt[0])
            t_pct = robust_float(t_pt[0])
            tol_pct = max(0.02 * abs(t_pct), 0.005)

            if abs(p_pct - t_pct) <= tol_pct:
                score += 0.5
            elif abs(p_pct - t_pct) <= tol_pct * 2:
                score += 0.2
        except (ValueError, TypeError):
            pass

        # 2. Verify Absolute Value if present (Index 1)
        if len(t_pt) > 1 and len(p_pt) > 1:
            try:
                p_val = robust_float(p_pt[1])
                t_val = robust_float(t_pt[1])
                tol_val = max(0.05 * abs(t_val), 0.01)

                if abs(p_val - t_val) <= tol_val:
                    score += 0.5
            except (ValueError, TypeError):
                pass

    # Normalize score across the number of pie slices
    return score / max(1, len(target_ser))

In [4]:
def dynamic_series_reward_func(completions, target_specs, **kwargs):
    """Reward 5: Master routing calculation for the `ser` array based on chart subtype."""
    rewards = []

    for comp, target_str in zip(completions, target_specs):
        parsed = extract_json(comp)
        target = target_str if isinstance(
            target_str, dict) else json.loads(target_str)

        if not parsed or "panels" not in parsed or "panels" not in target:
            rewards.append(0.0)
            continue

        overall_score = 0.0
        target_panel_count = len(target["panels"])

        for p_panel, t_panel in zip(parsed["panels"], target["panels"]):
            chart_type = t_panel.get("topo", {}).get("type", "")
            panel_score = 0.0

            if chart_type == "bar":
                panel_score = evaluate_bar_series(p_panel, t_panel)
            elif chart_type == "line":
                panel_score = evaluate_line_series(p_panel, t_panel)
            elif chart_type == "box":
                panel_score = evaluate_box_series(p_panel, t_panel)
            elif chart_type == "histogram":
                panel_score = evaluate_histogram_series(p_panel, t_panel)
            elif chart_type == "scatter":
                panel_score = evaluate_scatter_series(p_panel, t_panel)
            elif chart_type == "pie":
                panel_score = evaluate_pie_series(p_panel, t_panel)

            overall_score += panel_score

        # NORMALIZE BY PANEL COUNT:
        # Ensures multi-panel charts have the same maximum reward ceiling (1.0) as single-panel charts
        normalized_total_score = overall_score / max(1, target_panel_count)
        rewards.append(normalized_total_score)

    return rewards

In [ ]:
import json

# 1. Define Ground Truth
target_spec = {
    "title": "Usage Frequency of Transportation Modes",
    "panel_count": 2, "panel_layout": [2, 1],
    "legend": {"on": 0},
    "panels": [
        {
            "panel_idx": 0, "panel_type": "bar", "title": "Usage Frequency of Transportation Modes", "has_errorbar": 0,
            "topo": {"type": "bar", "sub": "horizontal"},
            "axes": [
              {"name": "x_axis", "scl": "linear", "dom": [0.0, 45]},
              {"name": "y_axis", "scl": "categorical", "dom": [
                  "Walk", "Train", "Bus", "Car", "Bike"]}
            ],
            "ser": [{"name": "bar_series_1", "color": "#ff7f0e", "symbol": "none", "y_ref": "x_axis", "data": [["Walk", 5], ["Train", 10], ["Bus", 20], ["Car", 30], ["Bike", 40]], "global_trend": "monotonic_increase"}]
        },
        {
            "panel_idx": 1, "panel_type": "bar", "has_errorbar": 0,
            "topo": {"type": "bar", "sub": "horizontal"},
            "axes": [
                {"name": "x_axis", "lbl": "Frequency of Use",
                 "scl": "linear", "dom": [0.0, 45]},
                {"name": "y_axis", "scl": "categorical", "dom": [
                    "Walk (New)", "Train (New)", "Bus (New)", "Car (New)", "Bike (New)"]}
            ],
            "ser": [{"name": "Frequency of Use", "color": "#ff7f0e", "symbol": "none", "y_ref": "x_axis", "data": [["Walk (New)", 7], ["Train (New)", 12], ["Bus (New)", 18], ["Car (New)", 25], ["Bike (New)", 35]], "global_trend": "monotonic_increase"}]
        }
    ],
    "chart_type": "bar"
}

target_str = json.dumps(target_spec)

# 2. Create Mock LLM Completions
# A. Perfect Match
completion_perfect = f"```json\n{target_str}\n```"

# B. Flawed Match (Missing a data point, wrong series name)
bad_spec = json.loads(target_str)
bad_spec["panels"][0]["ser"][0]["data"].pop()  # Hallucinated truncation
bad_spec["panels"][1]["ser"][0]["name"] = "Wrong Name"  # Wrong mapping
completion_flawed = f"```json\n{json.dumps(bad_spec)}\n```"

# C. Broken Syntax (Fails json.loads)
completion_invalid = "```json\n{ bad json syntax...\n```"

completions = [completion_perfect, completion_flawed, completion_invalid]
# Target is always the ground truth
target_specs = [target_str, target_str, target_str]

# 3. Test Reward Models
print("1. Format Reward (Expect [1.0, 1.0, -2.0]):")
print(format_reward_func(completions))

print("\n2. Schema Reward (Expect [0.5, 0.5, 0.0]):")
print(schema_enforcement_reward_func(completions))

print("\n3. Architecture Reward (Expect [1.0, 1.0, 0.0]):")
print(panel_architecture_reward_func(completions))

print("\n4. Axis Mapping Reward (Expect [1.0, 1.0, 0.0]):")
print(axis_mapping_reward_func(completions))

print("\n5. Dynamic Series Reward (Perfect > Flawed > Invalid):")
print(dynamic_series_reward_func(completions, target_specs=target_specs))

In [5]:
import json
import os
from datasets import Dataset

WORKSPACE_DIR = "/workspace/ImageToSpec_Stage1"
# Matches your updated image directory
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "final_balanced_images")


def load_grpo_dataset(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_data = []
    for item in data:
        raw_img_path = item["images"][0].replace("\\", "/")
        img_filename = os.path.basename(raw_img_path)
        local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)

        # ---> THE GRPO FIX <---
        # Extract the assistant's target JSON string from the messages array.
        # messages[0] is the user prompt, messages[1] is the assistant's correct JSON.
        target_spec_str = item["messages"][1]["content"]

        processed_data.append({
            "id": str(item["id"]),
            "messages": item["messages"],
            "images": [local_img_path],
            # <--- CRITICAL: Our reward functions read from this column!
            "target_specs": target_spec_str
        })

    return Dataset.from_list(processed_data)


# For GRPO, you only pass the training set to the RL algorithm so it can explore.
hf_dataset = load_grpo_dataset(os.path.join(
    WORKSPACE_DIR, "qwen25_vl_3b_train.json"))

print(f"Loaded {len(hf_dataset)} GRPO samples.")
print(f"Dataset Columns: {hf_dataset.column_names}")

Loaded 2010 GRPO samples.
Dataset Columns: ['id', 'messages', 'images', 'target_specs']


In [6]:
# ==========================================
# FORMAT DATASET FOR GRPO (UPDATED)
# ==========================================
def prepare_dataset_for_grpo(example):
    user_message = [msg for msg in example["messages"] if msg["role"] == "user"]
    assistant_message = [msg for msg in example["messages"] if msg["role"] == "assistant"]
    target_spec = assistant_message[0]["content"] if assistant_message else ""

    clean_user_message = []
    for msg in user_message:
        content = msg.get("content", "")
        if isinstance(content, str) and "<|image_pad|>" in content:
            # Split into proper multimodal content blocks
            text_part = content.replace("<|image_pad|>", "").strip()
            clean_content = [
                {"type": "image"},
                {"type": "text", "text": text_part},
            ]
            clean_user_message.append({"role": "user", "content": clean_content})
        else:
            clean_user_message.append(msg)

    return {
        "prompt": clean_user_message,
        "images": example["images"],
        "target_specs": target_spec,
    }

print("Formatting dataset for GRPO...")
hf_dataset = hf_dataset.map(
    prepare_dataset_for_grpo,
    remove_columns=hf_dataset.column_names # CRITICAL: Drop old columns to prevent collator confusion!
)

print("Columns available:", hf_dataset.column_names)

Formatting dataset for GRPO...


Map:   0%|          | 0/2010 [00:00<?, ? examples/s]

Columns available: ['images', 'target_specs', 'prompt']


In [7]:
from PIL import Image
import os

print("Validating image paths...")
bad = []
for i, ex in enumerate(hf_dataset):
    path = ex["images"][0]
    if not os.path.exists(path):
        bad.append((i, path))
    else:
        try:
            Image.open(path).verify()
        except Exception as e:
            bad.append((i, f"{path} — {e}"))

if bad:
    print(f"❌ Found {len(bad)} bad images:")
    for b in bad: print(" ", b)
    # Remove bad samples
    bad_indices = {b[0] for b in bad}
    hf_dataset = hf_dataset.filter(lambda _, i: i not in bad_indices, with_indices=True)
    print(f"✅ Dataset cleaned: {len(hf_dataset)} samples remaining")
else:
    print("✅ All images valid")

Validating image paths...
✅ All images valid


In [8]:
import json
with open("/workspace/ImageToSpec_Stage1/qwen25_vl_3b_train.json") as f:
    raw = json.load(f)

print(json.dumps(raw[0]["messages"][0], indent=2)) 

{
  "role": "user",
  "content": "<|image_pad|>\nExtract the Extended ChartSpec from this chart image. Return only the valid JSON."
}


In [ ]:
import torch
from unsloth import FastVisionModel, is_bf16_supported
from trl import GRPOConfig, GRPOTrainer

# 1. Load Model & Processor natively via Unsloth
model, processor = FastVisionModel.from_pretrained(
    "/workspace/ImageToSpec_Stage1/qwen3b_lora_sft_bf16_final/qwen3b_lora_sft_bf16_final",
    load_in_4bit = False, # Native bf16 for the A100!
    use_gradient_checkpointing = "unsloth", 
)

# ==========================================
# GRPO CONFIGURATION (UPDATED TO MATCH PAPER SCALE)
# ==========================================
training_args = GRPOConfig(
    output_dir="/workspace/unsloth_qwen3b_grpo",

    # Keeping generations at 8 to protect your 5090's 32GB VRAM
    num_generations=8,              
    num_train_epochs=1,             
    max_completion_length=3072,     

    # --- THE CRITICAL PAPER-ALIGNMENT FIXES ---
    learning_rate=4e-7,             # Matched exactly to the paper's 4x10^-7
    beta=0.001,                     # Matched exactly to the paper's 0.001 coefficient
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Memory & Optimization
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Global batch size = 8 prompts (64 total rollouts per step)
    gradient_checkpointing=True,
    bf16=is_bf16_supported(),
    optim="adamw_8bit",             

    logging_steps=5,
    save_steps=50,
)

# 3. Initialize GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[
        format_reward_func,
        schema_enforcement_reward_func,
        panel_architecture_reward_func,
        axis_mapping_reward_func,
        dynamic_series_reward_func
    ],
    train_dataset=hf_dataset,
    args=training_args,
)

# 4. Train and Save
print("Starting Unsloth GRPO Training on A100...")
trainer.train()

# Save the final RL-tuned weights
model.save_pretrained("/workspace/unsloth_qwen3b_grpo_final")
processor.save_pretrained("/workspace/unsloth_qwen3b_grpo_final")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2_5_Vl patching. Transformers: 5.8.0.dev0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.367 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


Starting Unsloth GRPO Training on A100...


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,010 | Num Epochs = 1 | Total steps = 2,010
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 148,611,072 of 4,051,845,120 (3.67% trained)
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=3072) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / schema_enforcement_reward_func / mean,rewards / schema_enforcement_reward_func / std,rewards / panel_architecture_reward_func / mean,rewards / panel_architecture_reward_func / std,rewards / axis_mapping_reward_func / mean,rewards / axis_mapping_reward_func / std,rewards / dynamic_series_reward_func / mean,rewards / dynamic_series_reward_func / std
5,0.005583,-0.578667,1.885200,1234.225000,269.200000,3072.000000,0.275000,557.138092,269.200000,1039.600000,5.582900,-0.950000,1.421467,0.000000,0.000000,0.350000,0.473822,0.350000,0.473822,-0.328667,0.584275
10,1.419056,-1.487000,0.991099,1198.800000,474.400000,2186.400000,0.125000,955.588342,474.400000,1681.600000,1419.056059,-1.550000,0.800408,0.000000,0.000000,0.137500,0.254905,0.150000,0.266803,-0.224500,0.462425
15,0.049632,-1.025917,1.936957,1108.350000,296.800000,2659.600000,0.125000,808.217151,296.800000,2212.400000,49.631730,-1.400000,1.224672,0.000000,0.000000,0.175000,0.337513,0.200000,0.408224,-0.000917,0.388160
20,0.017356,0.102000,1.772524,980.925000,212.000000,2624.000000,0.125000,692.951208,212.000000,1354.000000,17.356482,-0.875000,1.012540,0.062500,0.051755,0.375000,0.337513,0.375000,0.337513,0.164500,0.241854
25,0.014020,-0.820633,1.816613,794.450000,236.200000,2376.600000,0.050000,673.621436,236.200000,1796.400000,14.020103,-1.250000,1.121121,0.050000,0.087110,0.225000,0.370312,0.175000,0.337513,-0.020633,0.269271
30,0.179279,-0.370730,2.180001,915.675000,235.600000,2497.800000,0.075000,737.175012,235.600000,1487.800000,179.279445,-0.950000,1.421467,0.012500,0.035355,0.350000,0.473822,0.325000,0.451951,-0.108230,0.396507
35,0.037045,-0.852750,1.904670,1197.825000,242.800000,3072.000000,0.175000,794.428613,242.800000,1609.600000,37.045295,-1.250000,1.300470,0.000000,0.000000,0.225000,0.430095,0.250000,0.433490,-0.077750,0.504040


[transformers] Both `max_new_tokens` (=3072) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=3072) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=3072) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=3072) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (htt